<a href="https://colab.research.google.com/github/anirbanghoshsbi/.github.io/blob/master/work/indicator/temp/impulse_work_219.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyotp --q
!pip install smartapi-python==1.4.1 --q
!pip install logzero --q

In [2]:

# package import statement
from SmartApi import SmartConnect #or from SmartApi.smartConnect import SmartConnect
import pyotp
from logzero import logger
import time
import os
import urllib
import json
import pandas as pd
import datetime as dt

api_key = 'xOHnB7MG'
username = 'M55123447'
pwd = '1471'
smartApi = SmartConnect(api_key)
try:
    token = "GJZACUQI2TTAIBHBA34XNFJURQ"
    totp = pyotp.TOTP(token).now()
except Exception as e:
    logger.error("Invalid Token: The provided token is not valid.")
    raise e

correlation_id = "abcde"
data = smartApi.generateSession(username, pwd, totp)

if data['status'] == False:
    logger.error(data)

else:
    # login api call
    # logger.info(f"You Credentials: {data}")
    authToken = data['data']['jwtToken']
    refreshToken = data['data']['refreshToken']
    # fetch the feedtoken
    feedToken = smartApi.getfeedToken()
    # fetch User Profile
    res = smartApi.getProfile(refreshToken)
    smartApi.generateToken(refreshToken)
    res=res['data']['exchanges']
#Download Nifty50 Index Data
params = {
           "exchange": "NSE",
           "symboltoken": '99926000',
           "interval": "ONE_DAY",
           "fromdate": (dt.datetime(2025, 1, 1).strftime('%Y-%m-%d %H:%M')),
           "todate": (dt.datetime.today().strftime('%Y-%m-%d %H:%M'))
         }
nifty_data = smartApi.getCandleData(params)
nifty_data_format= pd.DataFrame(nifty_data["data"],
                               columns = ["Date","Open","High","Low","Close","Volume"])
nifty_data_format.set_index("Date",inplace=True)
nifty_data_format.index = pd.to_datetime(nifty_data_format.index)
nifty_data_format.index = nifty_data_format.index.tz_localize(None)

In [3]:
df = nifty_data_format.copy()

# --------------------------
# Moving Average
# --------------------------
df["ma20"] = df["Close"].rolling(20).mean()

# --------------------------
# EMA calculation
# --------------------------
df["ema12"] = df["Close"].ewm(span=12, adjust=False).mean()
df["ema26"] = df["Close"].ewm(span=26, adjust=False).mean()
df["ema13"] = df["Close"].ewm(span=13, adjust=False).mean()
df['ema50'] = df['Close'].ewm(span=50, adjust=False).mean()
# --------------------------
# MACD
# --------------------------
df["macd"] = df["ema12"] - df["ema26"]

# MACD Signal
df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()

# MACD Histogram
df["macd_hist"] = df["macd"] - df["macd_signal"]

In [4]:
df["ema_slope"] = df["ema13"].diff()
df["macd_slope"] = df["macd_hist"].diff()

df["impulse"] = "blue"

df.loc[
    (df["ema_slope"] > 0) & (df["macd_slope"] > 0),
    "impulse"
] = "green"

df.loc[
    (df["ema_slope"] < 0) & (df["macd_slope"] < 0),
    "impulse"
] = "red"

In [5]:
df["ret_10d"] = df["Close"].shift(-10) / df["Close"] - 1

In [6]:
long_cond = (df["Close"] > df["ma20"]) & (df["impulse"] == "green")

long_returns = df.loc[long_cond, "ret_10d"]

In [7]:
short_cond = (df["Close"] < df["ma20"]) & (df["impulse"] == "red")

short_returns = -df.loc[short_cond, "ret_10d"]

In [8]:
summary = pd.DataFrame({

    "count":[len(long_returns),len(short_returns)],

    "mean_return":[
        long_returns.mean(),
        short_returns.mean()
    ],

    "median_return":[
        long_returns.median(),
        short_returns.median()
    ],

    "win_rate":[
        (long_returns>0).mean(),
        (short_returns>0).mean()
    ]

}, index=["Long","Short"])

print(summary)

       count  mean_return  median_return  win_rate
Long      89    -0.002619      -0.004025  0.438202
Short     73     0.000222      -0.007976  0.452055


#BACKTEST
Close > MA20.
Impulse = Green
Entry on EOD

Exit on EOD
Close Below 20 EMA

In [9]:
df["entry_signal"] = (
    (df["Close"] > df["ma20"]) &
    (df["impulse"] == "green")
)

In [10]:
#df["exit_signal"] = df["Close"] < df["ma20"]

In [11]:
#df["exit_signal"] = df["Close"] < df["ema13"]

In [12]:
df["exit_signal"] = df["Close"] < df["ema50"]

In [13]:
position = 0
entry_price = 0

trades = []

for i in range(len(df)):

    price = df["Close"].iloc[i]

    if position == 0:

        if df["entry_signal"].iloc[i]:
            position = 1
            entry_price = price
            entry_date = df.index[i]

    elif position == 1:

        if df["exit_signal"].iloc[i]:

            exit_price = price
            exit_date = df.index[i]

            ret = (exit_price / entry_price) - 1

            trades.append({
                "entry_date": entry_date,
                "exit_date": exit_date,
                "entry_price": entry_price,
                "exit_price": exit_price,
                "return": ret
            })

            position = 0

In [14]:
trades_df = pd.DataFrame(trades)

In [15]:
stats = {
    "trades": len(trades_df),
    "mean_return": trades_df["return"].mean(),
    "median_return": trades_df["return"].median(),
    "win_rate": (trades_df["return"] > 0).mean(),
    "max_gain": trades_df["return"].max(),
    "max_loss": trades_df["return"].min(),
}

print(stats)

{'trades': 14, 'mean_return': np.float64(0.001792936786521577), 'median_return': -0.004781053204209351, 'win_rate': np.float64(0.42857142857142855), 'max_gain': 0.06466111267095465, 'max_loss': -0.015065766610149911}


In [16]:
trades_df

,entry_date,exit_date,entry_price,exit_price,return
0,2025-01-31,2025-02-03,23508.40,23361.05,-0.006268
1,2025-02-04,2025-02-10,23739.25,23381.60,-0.015066
2,2025-03-18,2025-03-19,22834.30,22907.60,0.003210
3,2025-03-20,2025-04-04,23190.65,22904.45,-0.012341
4,2025-04-15,2025-07-25,23328.55,24837.00,0.064661
5,2025-08-18,2025-08-26,24876.95,24712.05,-0.006629
6,2025-09-03,2025-09-04,24715.05,24734.30,0.000779
7,2025-09-05,2025-09-08,24741.00,24773.15,0.001299
8,2025-09-09,2025-09-25,24868.60,24890.85,0.000895
9,2025-10-06,2026-01-08,25077.65,25876.85,0.031869
